# 寶石方塊 Bejeweled Blitz 自動遊玩

這份筆記本包含程式與按鈕辨識資料，可以單獨複製到另一台電腦。

## 操作方式

1. 使用 Windows 和 Chrome，先完成 Facebook 登入。
2. 在 VS Code 開啟本檔，右上角選擇可用的 Python 核心。
3. 按「全部執行」。最後一格會啟動程式，自動開啟遊戲、按開始、交換寶石，並接著玩下一局。
4. 按 **F8** 停止，也可以按筆記本工具列的「中斷」。
5. 停止後若要再玩，重新執行最後一格即可。

執行 .ipynb 需要可用的 Jupyter／ipykernel 核心環境；遊戲程式本身使用 Python 標準函式庫與 Windows 內建功能。

保持棋盤和按鈕完整顯示。程式切到其他視窗時會暫停，遊戲倒數可能仍繼續。診斷記錄會寫在核心目前工作目錄的 blitz-last-run.log。

## 程式流程

擷取畫面 → 辨識 8×8 棋盤 → 寶石轉成編號 → 優先觸發火焰 → 交換 → 重新擷取。

編號為紅 1、橘 2、黃 3、綠 4、藍 5、紫 6、白 7，0 代表辨識不明。F 表示火焰標記。火焰外觀判斷和高分策略仍可能受動畫影響。


## 1. Windows 畫面擷取
使用內建 GDI/GDI+ 取得畫面像素。


In [ ]:
"""Windows screen/image pixels using only Python's standard library."""

import ctypes as ct
from ctypes import wintypes as wt


class BitmapInfo(ct.Structure):
    _fields_ = [
        ("size", wt.DWORD), ("width", wt.LONG), ("height", wt.LONG),
        ("planes", wt.WORD), ("bits", wt.WORD), ("compression", wt.DWORD),
        ("image_size", wt.DWORD), ("xppm", wt.LONG), ("yppm", wt.LONG),
        ("used", wt.DWORD), ("important", wt.DWORD),
    ]


class Frame:
    def __init__(self, data, width, height, stride=None, offset=0):
        self.data = data
        self.width, self.height = width, height
        self.stride = stride or width * 4
        self.offset = offset

    def pixel(self, x, y):
        index = self.offset + y * self.stride + x * 4
        blue, green, red = self.data[index:index + 3]
        return red, green, blue

    def crop(self, x, y, width, height):
        if min(x, y) < 0 or min(width, height) <= 0 or x + width > self.width or y + height > self.height:
            raise ValueError("擷取範圍超出圖片。")
        return Frame(self.data, width, height, self.stride, self.offset + y * self.stride + x * 4)


class Capture:
    def __init__(self):
        self.user = ct.WinDLL("user32", use_last_error=True)
        self.gdi = ct.WinDLL("gdi32", use_last_error=True)
        self.user.GetDC.argtypes = [wt.HWND]
        self.user.GetDC.restype = wt.HDC
        self.user.ReleaseDC.argtypes = [wt.HWND, wt.HDC]
        self.gdi.CreateCompatibleDC.argtypes = [wt.HDC]
        self.gdi.CreateCompatibleDC.restype = wt.HDC
        self.gdi.CreateDIBSection.argtypes = [
            wt.HDC, ct.POINTER(BitmapInfo), wt.UINT, ct.POINTER(ct.c_void_p), wt.HANDLE, wt.DWORD,
        ]
        self.gdi.CreateDIBSection.restype = wt.HBITMAP
        self.gdi.SelectObject.argtypes = [wt.HDC, wt.HANDLE]
        self.gdi.SelectObject.restype = wt.HANDLE
        self.gdi.BitBlt.argtypes = [
            wt.HDC, ct.c_int, ct.c_int, ct.c_int, ct.c_int,
            wt.HDC, ct.c_int, ct.c_int, wt.DWORD,
        ]
        self.gdi.DeleteObject.argtypes = [wt.HANDLE]
        self.gdi.DeleteDC.argtypes = [wt.HDC]
        self.screen = self.memory = None
        self.monitors = [dict(zip(
            ("left", "top", "width", "height"),
            (self.user.GetSystemMetrics(i) for i in (76, 77, 78, 79)),
        ))]

    def __enter__(self):
        self.screen = self.user.GetDC(None)
        self.memory = self.gdi.CreateCompatibleDC(self.screen) if self.screen else None
        if not self.memory:
            self.__exit__(None, None, None)
            raise OSError("無法建立 Windows 螢幕擷取，請在已登入的 Windows 桌面執行。")
        return self

    def __exit__(self, *args):
        if self.memory:
            self.gdi.DeleteDC(self.memory)
        if self.screen:
            self.user.ReleaseDC(None, self.screen)
        self.screen = self.memory = None

    def grab(self, region):
        width, height = region["width"], region["height"]
        if width <= 0 or height <= 0:
            raise ValueError("擷取大小無效。")
        info = BitmapInfo()
        info.size, info.width, info.height = ct.sizeof(info), width, -height
        info.planes, info.bits = 1, 32
        pointer = ct.c_void_p()
        bitmap = self.gdi.CreateDIBSection(self.screen, ct.byref(info), 0, ct.byref(pointer), None, 0)
        if not bitmap:
            raise ct.WinError(ct.get_last_error())
        old = self.gdi.SelectObject(self.memory, bitmap)
        try:
            if not old:
                raise OSError("無法選取擷取點陣圖。")
            if not self.gdi.BitBlt(
                self.memory, 0, 0, width, height, self.screen,
                region["left"], region["top"], 0x40CC0020,
            ):
                raise ct.WinError(ct.get_last_error())
            self.gdi.GdiFlush()
            return Frame(ct.string_at(pointer, width * height * 4), width, height)
        finally:
            if old:
                self.gdi.SelectObject(self.memory, old)
            self.gdi.DeleteObject(bitmap)


def read_image(path):
    """Decode PNG/JPEG with the GDI+ library included in Windows."""
    class Startup(ct.Structure):
        _fields_ = [("version", wt.UINT), ("callback", ct.c_void_p),
                    ("thread", wt.BOOL), ("codecs", wt.BOOL)]

    class Data(ct.Structure):
        _fields_ = [("width", wt.UINT), ("height", wt.UINT), ("stride", ct.c_int),
                    ("format", ct.c_int), ("scan", ct.c_void_p), ("reserved", ct.c_size_t)]

    class Rect(ct.Structure):
        _fields_ = [(name, ct.c_int) for name in ("x", "y", "width", "height")]

    dll = ct.WinDLL("gdiplus")
    dll.GdiplusStartup.argtypes = [ct.POINTER(ct.c_size_t), ct.POINTER(Startup), ct.c_void_p]
    dll.GdiplusShutdown.argtypes = [ct.c_size_t]
    dll.GdipCreateBitmapFromFile.argtypes = [wt.LPCWSTR, ct.POINTER(ct.c_void_p)]
    dll.GdipGetImageWidth.argtypes = [ct.c_void_p, ct.POINTER(wt.UINT)]
    dll.GdipGetImageHeight.argtypes = [ct.c_void_p, ct.POINTER(wt.UINT)]
    dll.GdipBitmapLockBits.argtypes = [ct.c_void_p, ct.POINTER(Rect), wt.UINT, ct.c_int, ct.POINTER(Data)]
    dll.GdipBitmapUnlockBits.argtypes = [ct.c_void_p, ct.POINTER(Data)]
    dll.GdipDisposeImage.argtypes = [ct.c_void_p]
    token = ct.c_size_t()
    if dll.GdiplusStartup(ct.byref(token), ct.byref(Startup(1, None, False, False)), None):
        raise OSError("無法啟動 Windows 圖片解碼。")
    bitmap = ct.c_void_p()
    try:
        if dll.GdipCreateBitmapFromFile(str(path), ct.byref(bitmap)):
            raise ValueError("無法讀取圖片，請確認路徑和檔案格式。")
        width, height = wt.UINT(), wt.UINT()
        if dll.GdipGetImageWidth(bitmap, ct.byref(width)) or dll.GdipGetImageHeight(bitmap, ct.byref(height)):
            raise ValueError("無法取得圖片大小。")
        data = Data()
        rect = Rect(0, 0, width.value, height.value)
        if dll.GdipBitmapLockBits(bitmap, ct.byref(rect), 1, 0x26200A, ct.byref(data)):
            raise ValueError("無法讀取圖片像素。")
        try:
            pixels = b"".join(ct.string_at(data.scan + y * data.stride, width.value * 4) for y in range(height.value))
            return Frame(pixels, width.value, height.value)
        finally:
            dll.GdipBitmapUnlockBits(bitmap, ct.byref(data))
    finally:
        if bitmap:
            dll.GdipDisposeImage(bitmap)
        dll.GdiplusShutdown(token)


## 2. 寶石編號與火焰辨識
辨識顏色、火焰和棋盤位置。


In [ ]:
"""Gem IDs and flame detection from sampled RGB pixels, without pip packages."""

from colorsys import rgb_to_hsv


def hsv(pixel):
    hue, sat, value = rgb_to_hsv(*pixel)
    return hue * 180, sat * 255, value


def color_id(pixel):
    hue, sat, value = hsv(pixel)
    if sat <= 80 and value >= 145:
        return 7
    if sat < 100 or value < 95:
        return None
    if hue <= 8 or hue >= 172:
        return 1
    for limit, code in ((21, 2), (38, 3), (82, 4), (125, 5), (171, 6)):
        if hue <= limit:
            return code
    return None


def read_board(frame):
    rows = []
    for row in range(8):
        cells = []
        for col in range(8):
            counts = [0] * 8
            for y in range(8):
                for x in range(8):
                    px = int((col + 0.24 + x * 0.52 / 7) * frame.width / 8)
                    py = int((row + 0.24 + y * 0.52 / 7) * frame.height / 8)
                    code = color_id(frame.pixel(px, py))
                    if code is not None:
                        counts[code] += 1
            ranked = sorted(range(1, 8), key=lambda code: counts[code], reverse=True)
            best, second = ranked[:2]
            cells.append(best if counts[best] >= 20 and counts[best] - counts[second] >= 11 else None)
        rows.append(tuple(cells))
    return tuple(rows)


def flame_strength(frame):
    # A flame adds yellow light above and outside the ordinary gem silhouette.
    lit = 0
    total = 0
    for y in (0.04, 0.08, 0.12, 0.16, 0.20):
        for x in (0.16, 0.23, 0.30, 0.37, 0.63, 0.70, 0.77, 0.84):
            hue, sat, value = hsv(frame.pixel(int(x * frame.width), int(y * frame.height)))
            lit += 14 <= hue <= 38 and sat >= 70 and value >= 190
            total += 1
    return lit / total


def read_flames(frame, board):
    flames = set()
    for row in range(8):
        for col in range(8):
            if board[row][col] is None:
                continue
            x, y = col * frame.width // 8, row * frame.height // 8
            right, bottom = (col + 1) * frame.width // 8, (row + 1) * frame.height // 8
            if flame_strength(frame.crop(x, y, right - x, bottom - y)) >= 0.12:
                flames.add((row, col))
    return flames


def countdown_visible(frame):
    count = 0
    for y in range(20):
        for x in range(20):
            red, green, blue = frame.pixel(
                int((1 / 3 + x / 60) * frame.width),
                int((3 / 8 + y / 80) * frame.height),
            )
            count += green >= 250 and red <= 4 and blue <= 4
    return count > 14


def cyan(pixel):
    red, green, blue = pixel
    if blue < 100 or green < 70 or red > max(green, blue) * 0.62:
        return False
    hue, sat, value = hsv(pixel)
    return 83 <= hue <= 107 and sat >= 100 and value >= 100


def timer_active(frame):
    hits = sum(cyan(frame.pixel(x * frame.width // 8, y * frame.height // 4))
               for x in range(8) for y in range(4))
    return hits > 8


def find_board(frame):
    # Search for the full cyan bar during the opening countdown.
    tested = set()
    for y in range(2, frame.height, 4):
        start = None
        for x in range(0, frame.width + 4, 4):
            lit = x < frame.width and cyan(frame.pixel(x, y))
            if lit and start is None:
                start = x
            if lit or start is None:
                continue
            left, right = start, min(x, frame.width)
            start = None
            if right - left < 150:
                continue
            while left > 0 and cyan(frame.pixel(left - 1, y)):
                left -= 1
            while right > left and not cyan(frame.pixel(right - 1, y)):
                right -= 1
            pending = [((left + right) // 2, y)]
            visited = set()
            top, bottom = y, y
            while pending:
                px, py = pending.pop()
                if (px, py) in visited or not (0 <= px < frame.width and 0 <= py < frame.height):
                    continue
                visited.add((px, py))
                if not cyan(frame.pixel(px, py)):
                    continue
                left, right = min(left, px), max(right, px + 1)
                top, bottom = min(top, py), max(bottom, py)
                pending.extend(((px-1, py), (px+1, py), (px, py-1), (px, py+1)))
            if not 14 <= (right - left) / (bottom - top + 1) <= 29:
                continue
            key = (left, top, right)
            if key in tested:
                continue
            tested.add(key)
            side = round((right - left) * 1.013)
            board_x = left - round(side * 0.008)
            board_y = top - round(side * 0.018) - side
            if min(board_x, board_y) < 0 or board_x + side > frame.width or board_y + side > frame.height:
                continue
            board = read_board(frame.crop(board_x, board_y, side, side))
            gems = [gem for row in board for gem in row if gem is not None]
            if len(gems) >= 60 and len(set(gems)) >= 5:
                return board_x, board_y, side, side
    return None


## 3. 開局按鈕辨識
下方資料是 PLAY、PLAY AGAIN、NO THANKS 按鈕的取樣像素，已壓縮內嵌。資料格預設收合，接下來的程式會將它還原並進行比對。


In [ ]:
import base64
import json
import zlib

# 四種按鈕的取樣資料，使用 Python 內建功能解壓縮。
BUTTON_DATA = (
    'eJztXNmO5bYR/Zd+Jgzui3/FaBiDwEiMjCcGxg8JAv97auFSlChd9To9zjUwRqmvRJE8VcVisY5++u/Dl0+//fLw'
    '48OXf/38xz8+ffnn15+//vbp8+cH9fDp6++//O2Phx/9D8FEY4t21oasY1APv//6718+f3348aefolY2Kx8f1U/R'
    'KptUsCDmqIwy3tzFby9Gp2xUkUStnFY+gRhKx83kokxIyhoNV9YHZZ1RNnj8LVgFCAd3lz+WnADH0EACiL1XluBG'
    'IDNjXLRKoAR4i7VemVQ2qN7l70SOSbnULjRYtlaGLDpnsvPwUTp6l18sC0hfCV1rwPhD8xZWw5VdugJYCIprt8FK'
    'YcZV8Kr+wysPyuji6lVT45un5qu5U9CG9XnV4vTU9IsHyfRnkldh2aXjBuYOQfBicrrdQizK2wsj/NBKEBBZu3pm'
    'UoKLc3fxpdPV3Noz4J+nflLX56E3KYDV0KAdugadc/ngzknnZ63MTqWVTn0IJZgGhSGg6RN4UQt8wbDj1bTgxF2c'
    '6IFU5WPFppGaMXZoy+b1u2Z4N2+Ws5SyinF9H4bdK4X+aEowj8EBnqbr67dRgouuALY4uTfn4M863O4qRlHmivPe'
    '6sCkY2DehO9+wMfe5Q10AAJBjAPTc+RS59/qzS/bXkdUAVPK/qfjh6699KQLuqqAPX0P2a8BxXP73m165KwCCE0O'
    '9FhhJfDcjaBwlNaafSOopagEzpx3BF0LKIxlXEADQA9MMfuXyel8KmbvqgUa1B8XLpt3WgD+gVZa5/YT4R0Hhj6/'
    'px6QdbqV9oGHMOTTwg7ODTLHrR8rATk+9J2mGiymXUq8pYwvQOwNdOAufxj5ju5fWb6j+1eWJaSwQYA/Z0quwxqB'
    'q9hzFvy7/I1liD90W91tRJmOSPDMy0Gcj3+GyE+30EDcjtESRDX+Lr6n2BAgZDwhQwchEfYIsAHwM3YRgzUVqY1A'
    'baDB1pMzexffVSwk0jEynlmphA4Tt2VGefP4+KfqB9W/f/70n58//f3Tr1/EGbX7wWrncslOB5198tMhtYmAL+xo'
    'yAnjW0FPOEoPeIpmsr9f/FUuDOqP04oWYrD2UhTuRS2mZYxRnCnAFA1mi+imjMekiTasqSinUniJCNvVKmaH2ptp'
    'JSlZYb84x6ALaaOhTTIlkxz0PlLXYFNqyDHR5trhVtv0k3mQdM9U4MYTvJp19rFulaEVPgOGNUjRgH1sw8UTGz/S'
    'JKZ6SZ6Go5RrENN0mizrrYw7cYpHgkD398Gk0Fl1bj3D3AmFTbSdxiyKS3SVOF1rS4XPFPxLavOCM605nwMzCM/C'
    'hNN8gkQJOtvmEzP5mZUgUHrT90Tj0II1nABaUCm9kphRJEWLsPDsxQSqozKlI1JQoLiudsc0PcrgJa3K7Mgwt6EK'
    'LXYYfcIT1GdQGBhjoRYxtwI/aVOV0MJUaJomjWoFF6Qh2uECaTRNNaoYPMY6pzH9AhdWXrj9RZS3edmA4abZDAq/'
    'lPsZcP3l7phiuaM0eFwNoiqxu2tVfLUoHDEPxuGsFM4g4ah4hih7jZORqo04eJ6OyidYHc55eiUxDDGjKyF0khlY'
    'epVV5j5EFG3tTkM4WxR5iB5F9h+JRhXFCPMYOesAzmGfhYy6wVlHOrxi3UBwUWacWGd8xWzI8u+e5KYlrR3Z/vxe'
    '2R9Dcm46qZvekjrHzWBhCnJT8jExMNZSwZT4ofGinwxbT/0GYOYXgxmahTYEK5rpFM0oZrUINI1A0wuk6CyKrJ4W'
    'Fc82M/9dIms7sq19295rn4bmGNcYrpiElAacZhjnjKdXDqNu/U3w7CCGAWIc6liGRTrq/x5QMSFyotDlSkC7yRQB'
    'ohUmFmZA/QGgfgWomwF1M6Bm9KeCuwHUrACV49XDVlMHtyLaVlQHgxHBRKgp8KdAug01pivopQiQpt9umuvm9mGx'
    'c+AjtJgKCej4RprtmRPevGNyw8Jwz4Y4Gaxwy5u7bHfkk3tAACDSrQdOm95ogZWvEdVVI45LI55QxspS0E23hx9i'
    'tMi76yuYX1SGm/bMYXFpQSSFxXhKs7HqGeXNO7auuloMWmRC70twepxwnFBfgYaQh8/HIEznoL2vvt3Uhe/GUBRD'
    'KLv7hcL1HrzjOTQGNLEI5bIddrOEfXP1BOBbEEIK5fZWXaisGC7QujAoRxUvbeHDoNzY3gP09RSww/+VVpwEdZhg'
    'C/tp59M01zctZtq0TACNJqSIHgrDHv57XL/FeH3wegxMsfscRHvMTXC7PjeRHYZTox4Xfki9t+3ck7DFs2nHSSkY'
    'V7ttHjAmFDHytg0k0KGaS4TB+I3B1QGI5vAkNkTs9r5xPIVF/TI0f1T70l5FmyVuwsAUV7Gg5HpLAzuMJR1qMbak'
    'xTn6dBssObC48wOYkeNT1SVSV9B8+j2nbsOVI3XwA+fQ3yEhv/oOjrNtC2hg34LOgrZHaEwNsxn06yO4BrtBdNYD'
    'zWOgaYi+j3mghoYMa3f36bMqKNIDg+ays+j3wHLb2jDUSJVMRo94NVat3c6FG/q/tPNWGMc5fmnn6J+w1K9Vc5il'
    'ZW+iDsw4cAS2X/QaqK5WI7CFbtFOZ2gvnMTSwlHD+2BtE9HnKXI6sKlLbMKYSiptNy7iuFtIv4bohjgM0qUhlmGm'
    'ZozHLe14aHrQQxyPBd8nJ4zJCeKxMZF4ijBEsxC7lcMmdbQbhzicTbCrnkkjFYo5jHTAhgqeWzAz0LIwf6FG/h8C'
    'uHgAnB7A6QGccFH9r2UJ3MBbAifmWiBQboni3rRUCHcLuHQbuCbiysAR9gQWYniWWf54uJkFbmLwZTVRw5dOtheX'
    'AORbYlq2INp1K90RPZO4iVGMx6JXXqW+oo/NI+4eOC8PyMFugm6h5Uelttl46VnC8ljhKY/B/h1E1joP8RmnsGB/'
    'k9pWXve83JRQjXNeoechLCVmOU2MjLKaWqZNa+6p5cyJ3ZbGoKRzlBfp8Je4b2Bqenrp6I3sZeu93Y1qDHZMwYwl'
    'VvzDXNMP4sSINMKrvrhDrOvu4rcScacEMNXKaNyIp44XQDydFnda83xY7Kb/vDwsdnRgQSYPO2zwjq4nvRLG7agD'
    'MVGRiL2L7ymiXeOulvOR4JcdYsF4UXoEAjAIx2p2uUy7YVsLtNmawZOjxef7xXd2AbtOF+uZsMGUVqwH4A6PZThH'
    'hhsa3KfxySSmVEzLKn2AEdwvXl0nPHFuiKmAhx2xll4mDLlrNubb9/J+8W4XEQI4ZGTedeD/9+I9dGCbc7v4E4av'
    'xLrlTQx0M5l6XjW/C0tfiFB164Dn5LGr48L6Gzzd4COLy6PEEqrG/Dy90dS8Kcdumkq0rJ0PVfZD+c5VAvM7WI/G'
    'Vcjb3C+WdcU6CXMjmAsmQvl5BzGuIWaav4H0RY2gswmn63ufqd0nKvHMFmmbX5YW8h2qxNn8HI/06iRcVYlnAsNn'
    '092Or7b4TKW4qiHvohPkp+th+4svNoM5+ckFYufznhe5zliz4Jtvwe9T+Hj+Xto1o/HfGAbmVXUraLja9U1vE85c'
    '6lzqnS/s/R0LgtvfSV8gK3WBvDpvVOeEJzC8N3xVuNYqgRzoehT+THEezqUfcF6cbgTrxLXjfGi4fgev7I3JfXAP'
    '2CfVspjN3w1VIdSSCKz0wCr7ctbXkyEdIz7dNn2/A9Q9YNUyL0it5vwFU74W3wrgu/i24h2371OUuBFXlupHsXoL'
    'brhD+LFEKqMJdRVIVHfKpL1UKDXvZpGpfOl4tbmLb4wXFkyX9pGZjHShFjDJC1hjiQS5RjFh6YKhWO0uvpt4jBZW'
    'rdKRZMItAEel+JjrLtSPnDdb5118NxEPI5nOabJCyCRSW/LydBJd2cop2mRMjtNJNDH/bOAKhJg7rRUD5w/EuL1f'
    'vMIF7NKxWAn/jAdY4BNGTcOStZxes7woY+154yxVqjKzNwtX0XJdO1Y3Y81l4gQESm13iBtTTDXUL20R0dZXxsEg'
    '9vatPxXX8iQ4RXdXQvBMFg5qfNeLr/xyn7m6mu+c6Mi+JRe2dGRTv17meq9D67WPhEQbEQaqWFPPTAxALrYICZ8h'
    'vhLPEkZOWDbNPAaQsGiNq6mZ62lnLDYE1kv45y56atI8WQxDZFobVyFrkjvZwVR+SAhCTsRI7sWythJnoiWZfGIk'
    'yhqrF5bI1WJo8mm+1T9SdoGLtjCNFGv5WQqD/5sSMZu7HJschezpfrdrU5Ns2nu7HER/qMqr9llTn3vJoWtjpOIy'
    'DrSw0tC2eeDxujZvZjAJ9wb3bizla2xWWye0U1j1RAXqZKEo5CK4cHYmJNsVIdmuaVATs/Eiy7HTVtOW5TjRVjn8'
    'fRptVZBBXaeIxtLnK8ZB1h9o+onWGt+MprxCk74KQa42XnKL3YFUZFlOO3pffQYXI92T4BNR+ITP6lXPUVPp5IKL'
    'OaPtDziPC4yZR0FpSUzZ70d9jnNzGAuck5Y4778JssHZYdkv+/VXxtnLDwrECzjDnKOx+dZLudrOUxNbCewWZ7lM'
    'HX9WwArMwxFPddO3OFty+xTB5nTA0EdAzQ12LVId8CMY9UBEfi3BDHTz4HELgx4f1yCDZl8obEoCXanp8szh6azm'
    'N7DotZfeg3xkzWGy5r2v3lvzkWVugjP6Wkuup0wT6XUT4M3WLSx6M4YjdrodXistqcpPMegJWk9U5unsu2JuIThX'
    '7NBfwaTTNaC3H5c4BnplzkhG7tYs18YkrFnPmK8tE4J+VI7JFpppr78OsrkLKw8G03kzAkG1vmjM5tiYQQHxvHTB'
    'WsbDPIvdeNywlinmp0/X8GA1fdu4Mt9ek2dDX2d6BvFzM2GDpbI3mOJrTv0my4Ywsa6Z69ySICVufhCtdvLUaW8B'
    'Mdwf8GxKBs4Rz83siUaCfiRYgBiqwwBSn8xOUj1kKEteskGm0zX26huB3D6a77mSQjBSwwGn8wRhOwhRa6r59Svc'
    '8dPa13ZBrW2kzOQ6lZueyR4MOqTkoA52lh/6KMiMGM4tqJ4rdvIxqOnNyXJn+PKnQPtwVjy5PpxTZNfs1EtU4/Or'
    '+eMFAx1YUzHjoxcL9CAYXzBfd918BbICNQGyw8WgbGPrDwTr4HMeueQneHVY7w4REARk3XUA1y/6QNCiKiYMdM6Q'
    'HcZ3QEBe8iGfh6wlegbzBpnz+kEMtrsivfFKwle9BFmW1l9URKDsWE/34vb2NHzxLVzTwDUNXDfrywZXM3A998US'
    '1wHmbe7r62I8DPD2dwKEsU50c/G9jybuuMrCE7M45u+MqxzHDT1vuGjsKQRlER7ZlU8SnkrSt90KuCBZr/jhCfwx'
    'XcPxHURwiL7ulzJ+lYKj/lz6DmHKOUq+bxFZJUtpYa6fxbxBJw4n5hdzTqJ+FbNFPzMn2agDgrJ4Jo7vWNJ+ebxn'
    'dIA6FlYEZTEQMT45apqLbTrxCEvKzMKGlb+FGoi0ivOYKOP8YTi8d5FE2I6jw2lKCSbuHx//fPwfPZfuoQ=='
)
MENU_REFERENCES = json.loads(zlib.decompress(base64.b64decode(BUTTON_DATA)))


In [ ]:
"""Recognize only the supplied PLAY / PLAY AGAIN / NO THANKS button images."""

import json
from pathlib import Path


def button_color(pixel):
    red, green, blue = pixel
    return max(red, blue) >= 100 and min(red, blue) >= 45 and green < 85 and min(red, blue) > green * 1.8


def button_boxes(frame, step=3):
    points = {
        (x, y) for y in range(0, frame.height, step) for x in range(0, frame.width, step)
        if button_color(frame.pixel(x, y))
    }
    boxes = []
    while points:
        seed = points.pop()
        pending = [seed]
        left = right = seed[0]
        top = bottom = seed[1]
        count = 0
        while pending:
            x, y = pending.pop()
            count += 1
            left, right = min(left, x), max(right, x)
            top, bottom = min(top, y), max(bottom, y)
            for neighbor in ((x-step, y), (x+step, y), (x, y-step), (x, y+step)):
                if neighbor in points:
                    points.remove(neighbor)
                    pending.append(neighbor)
        width, height = min(frame.width, right + step) - left, min(frame.height, bottom + step) - top
        if 60 <= width <= 460 and 14 <= height <= 120 and 1.9 <= width / height <= 7:
            if count * step * step >= width * height * 0.35:
                boxes.append((left, top, width, height))
    return boxes


def signature(frame):
    values = []
    for row in range(16):
        for col in range(40):
            x = min(frame.width-1, int((col + 0.5) * frame.width / 40))
            y = min(frame.height-1, int((row + 0.5) * frame.height / 16))
            values.append(frame.pixel(x, y))
    return values


def white(pixel):
    return min(pixel) >= 165 and max(pixel) - min(pixel) < 85


class MenuButtons:
    def __init__(self):
        self.references = MENU_REFERENCES

    def find(self, frame):
        matches = []
        for x, y, width, height in button_boxes(frame):
            left, top = max(0, x-4), max(0, y-4)
            crop = frame.crop(left, top, min(frame.width, x+width+4)-left, min(frame.height, y+height+4)-top)
            refined = button_boxes(crop, step=1)
            if not refined:
                continue
            dx, dy, width, height = max(refined, key=lambda box: box[2]*box[3])
            x, y = left+dx, top+dy
            pixels = signature(frame.crop(x, y, width, height))
            for reference in self.references:
                if abs(width / height / reference["aspect"] - 1) > 0.18:
                    continue
                samples = reference["pixels"]
                error = sum(abs(a-b) for p, q in zip(pixels, samples) for a,b in zip(p,q)) / (len(pixels)*3)
                text_union = sum(white(p) or white(q) for p,q in zip(pixels,samples))
                text_overlap = sum(white(p) and white(q) for p,q in zip(pixels,samples))
                overlap = text_overlap / max(1, text_union)
                if error < 33 and overlap > 0.48:
                    matches.append((error, reference["name"], (x+width//2, y+height//2)))
        if not matches:
            return None
        _, name, point = min(matches)
        return name, point


## 4. 交換策略與自動操作
列舉合法交換，優先觸發火焰，再比較其他走法。


In [ ]:
"""Open Bejeweled Blitz in Chrome and play successive rounds on Windows."""

import argparse
import ctypes
from ctypes import wintypes
import logging
import os
from pathlib import Path
import shutil
import subprocess
import time



SIZE = 8
COLORS = (1, 2, 3, 4, 5, 6, 7)
F2, F8 = 0x71, 0x77
LOG = logging.getLogger(__name__)
LOG_PATH = Path.cwd() / "blitz-last-run.log"


def runs(board):
    found = []
    for vertical in (False, True):
        for line in range(SIZE):
            cells = [(i, line) if vertical else (line, i) for i in range(SIZE)]
            start = 0
            while start < SIZE:
                row, col = cells[start]
                color = board[row][col]
                end = start + 1
                while end < SIZE:
                    row, col = cells[end]
                    if board[row][col] != color:
                        break
                    end += 1
                if color is not None and end - start >= 3:
                    found.append(set(cells[start:end]))
                start = end
    return found


def predict_value(board, matches, can_predict_cascades):
    # Use only visible gems when estimating cascades. New gems are unknown.
    value = 0
    for step in range(4 if can_predict_cascades else 1):
        if not matches:
            break
        removed = set.union(*matches)
        longest = max(map(len, matches))
        intersects = sum(map(len, matches)) > len(removed)
        special_bonus = 12 if longest >= 5 else 8 if intersects else 4 if longest == 4 else 0
        value += (2 * len(removed) + special_bonus) / (step + 1)
        for col in range(SIZE):
            kept = [board[row][col] for row in range(SIZE) if (row, col) not in removed]
            for row, gem in enumerate([None] * (SIZE - len(kept)) + kept):
                board[row][col] = gem
        matches = runs(board)
    return value


def best_move(board, flames=frozenset()):
    best = None
    best_value = (-1, -1)
    complete = all(None not in row for row in board)
    for row in range(SIZE):
        for col in range(SIZE):
            source = board[row][col]
            if source is None:
                continue
            for r, c in ((row + 1, col), (row, col + 1)):
                if r >= SIZE or c >= SIZE or board[r][c] in (source, None):
                    continue
                trial = [list(line) for line in board]
                trial[row][col], trial[r][c] = trial[r][c], trial[row][col]
                matches = runs(trial)
                if not any((row, col) in run or (r, c) in run for run in matches):
                    continue
                source_cell, target_cell = (row, col), (r, c)
                moved_flames = {
                    target_cell if cell == source_cell else source_cell if cell == target_cell else cell
                    for cell in flames
                }
                triggered = len(set.union(*matches) & moved_flames)
                # Explosion outcomes need new screen evidence, so avoid predicting their cascades.
                value = (triggered, predict_value(trial, matches, complete and not triggered))
                if value > best_value:
                    best = ((row, col), (r, c))
                    best_value = value
    return best


class Windows:
    def __init__(self):
        self.user = ctypes.WinDLL("user32", use_last_error=True)
        self.user.SetProcessDPIAware()
        self.user.GetForegroundWindow.restype = wintypes.HWND
        self.user.GetWindowTextW.argtypes = [wintypes.HWND, wintypes.LPWSTR, ctypes.c_int]
        self.user.GetWindowRect.argtypes = [wintypes.HWND, ctypes.POINTER(wintypes.RECT)]
        self.user.SetForegroundWindow.argtypes = [wintypes.HWND]
        self.user.ShowWindow.argtypes = [wintypes.HWND, ctypes.c_int]
        self.user.IsWindowVisible.argtypes = [wintypes.HWND]
        self.user.GetCursorPos.argtypes = [ctypes.POINTER(wintypes.POINT)]
        self.user.GetAsyncKeyState.argtypes = [ctypes.c_int]
        self.user.GetAsyncKeyState.restype = ctypes.c_short
        self.user.SetCursorPos.argtypes = [ctypes.c_int, ctypes.c_int]
        self.user.mouse_event.argtypes = [wintypes.DWORD] * 4 + [ctypes.c_size_t]

    def pressed(self, key):
        return bool(self.user.GetAsyncKeyState(key) & 0x8000)

    def check(self):
        if self.pressed(F8):
            raise KeyboardInterrupt

    def wait_in_game(self, window=None):
        while True:
            while not self.pressed(F2):
                self.check()
                time.sleep(0.015)
            while self.pressed(F2):
                self.check()
                time.sleep(0.015)
            self.check()
            active = self.game_window()
            if active is not None and (window is None or active == window):
                return active
            LOG.warning("這次 F2 已略過。請切回原本的 Chrome 遊戲分頁再按 F2。")

    def game_window(self):
        window = self.user.GetForegroundWindow()
        title = ctypes.create_unicode_buffer(512)
        self.user.GetWindowTextW(window, title, len(title))
        if "bejeweled blitz" not in title.value.lower() or "chrome" not in title.value.lower():
            return None
        return window

    def find_game_window(self):
        found = []
        callback_type = ctypes.WINFUNCTYPE(wintypes.BOOL, wintypes.HWND, wintypes.LPARAM)
        def visit(window, parameter):
            title = ctypes.create_unicode_buffer(512)
            self.user.GetWindowTextW(window, title, len(title))
            text = title.value.lower()
            if self.user.IsWindowVisible(window) and "bejeweled blitz" in text and "chrome" in text:
                found.append(window)
            return True
        callback = callback_type(visit)
        self.user.EnumWindows.argtypes = [callback_type, wintypes.LPARAM]
        self.user.EnumWindows(callback, 0)
        return found[0] if found else None

    def open_game(self):
        window = self.find_game_window()
        if window is None:
            candidates = [shutil.which("chrome")]
            for base in ("LOCALAPPDATA", "PROGRAMFILES", "PROGRAMFILES(X86)"):
                candidates.append(str(Path(os.environ.get(base, "")) / "Google/Chrome/Application/chrome.exe"))
            executable = next((p for p in candidates if p and Path(p).is_file()), None)
            if executable is None:
                raise ValueError("找不到 Google Chrome，請先在 Chrome 開啟 Bejeweled Blitz。")
            subprocess.Popen(
                [executable, "--new-window", "--start-maximized", "https://apps.facebook.com/bejeweledblitz"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        LOG.info("正在等待 Chrome 遊戲頁面。若 Facebook 要求登入，請完成登入。")
        deadline = time.monotonic() + 120
        while window is None and time.monotonic() < deadline:
            self.check()
            window = self.find_game_window()
            time.sleep(0.1)
        if window is None:
            raise ValueError("兩分鐘內未找到遊戲頁面，請確認已登入 Facebook。")
        self.user.ShowWindow(window, 3)
        self.user.SetForegroundWindow(window)
        return window

    def click(self, window, point):
        self.check()
        if self.game_window() != window:
            return False
        self.move_cursor(point)
        if self.game_window() != window:
            return False
        self.user.mouse_event(0x0002, 0, 0, 0, 0)
        try:
            time.sleep(0.06)
        finally:
            self.user.mouse_event(0x0004, 0, 0, 0, 0)
        return True

    def cursor(self):
        point = wintypes.POINT()
        if not self.user.GetCursorPos(ctypes.byref(point)):
            raise ctypes.WinError(ctypes.get_last_error())
        return point.x, point.y

    def window_region(self, window, desktop):
        bounds = wintypes.RECT()
        if not self.user.GetWindowRect(window, ctypes.byref(bounds)):
            raise ctypes.WinError(ctypes.get_last_error())
        left = max(bounds.left, desktop["left"])
        top = max(bounds.top, desktop["top"])
        right = min(bounds.right, desktop["left"] + desktop["width"])
        bottom = min(bounds.bottom, desktop["top"] + desktop["height"])
        if right <= left or bottom <= top:
            raise ValueError("Chrome 遊戲視窗在螢幕範圍外。")
        return dict(left=left, top=top, width=right - left, height=bottom - top)

    def move_cursor(self, point):
        if not self.user.SetCursorPos(*point):
            raise ctypes.WinError(ctypes.get_last_error())

    def swap(self, window, source, target):
        self.check()
        if self.game_window() != window:
            return False
        self.move_cursor(source)
        self.check()
        if self.game_window() != window:
            return False
        self.user.mouse_event(0x0002, 0, 0, 0, 0)
        try:
            for step in range(1, 5):
                time.sleep(0.012)
                self.check()
                if self.game_window() != window:
                    return False
                self.move_cursor(tuple(a + (b - a) * step // 4 for a, b in zip(source, target)))
        finally:
            self.user.mouse_event(0x0004, 0, 0, 0, 0)
        return True


def print_board(board, flames=frozenset()):
    for r, row in enumerate(board):
        print(" ".join(str(color or 0) + ("F" if (r, c) in flames else " ")
                       for c, color in enumerate(row)))


def describe(move):
    return " → ".join(f"第 {r + 1} 列第 {c + 1} 欄" for r, c in move)


def calibrate(win):
    LOG.info("切到 Chrome 的 Bejeweled Blitz。指向棋盤左上角，按 F2。")
    window = win.wait_in_game()
    left, top = win.cursor()
    LOG.info("已記錄左上角 (%d, %d)。指向棋盤右下角，再按 F2。", left, top)
    win.wait_in_game(window)
    right, bottom = win.cursor()
    width, height = right - left, bottom - top
    LOG.info("已記錄右下角 (%d, %d)。框選大小 %d × %d。", right, bottom, width, height)
    if min(width, height) < 160 or abs(width - height) > 0.08 * max(width, height):
        raise ValueError("框選範圍應為包含 8×8 格的正方形棋盤，請重新啟動。")
    LOG.info("按第三次 F2 待命，再用滑鼠按遊戲的開始按鈕。隨時按 F8 停止。")
    win.wait_in_game(window)
    return window, (left, top, width, height)


def play(dry_run, manual=False, wait_key=False, window=None):
    win = Windows()
    automatic = not (manual or wait_key or dry_run)
    if manual:
        window, bounds = calibrate(win)
    elif not automatic:
        LOG.info("切到 Chrome 的 Bejeweled Blitz，按一次 F2 待命，再用滑鼠開局。F8 停止。")
        window = win.wait_in_game()
        bounds = None
    else:
        window = window or win.open_game()
        bounds = None
    LOG.info("等待開局。F8 可隨時停止。")
    started = time.monotonic()
    deadline = started + (180 if automatic else 30)
    previous = last_move_board = None
    previous_flames = set()
    moves = 0
    idle_since = started
    last_timer_seen = None
    away = False
    region = None
    menus = MenuButtons() if automatic else None
    last_button_time = 0
    pending_button = None
    with Capture() as capture:
        while time.monotonic() < deadline:
            win.check()
            if win.game_window() != window:
                if not away:
                    LOG.info("已暫停。切回原本的 Chrome 遊戲分頁即可繼續。")
                    away = True
                previous = None
                time.sleep(0.05)
                continue
            if away:
                LOG.info("已切回遊戲分頁，繼續偵測棋盤。")
                away = False
                idle_since = time.monotonic()
            if bounds is None:
                search = win.window_region(window, capture.monitors[0])
                screen = capture.grab(search)
                if menus is not None:
                    button = menus.find(screen)
                    if button is not None:
                        name, point = button
                        now = time.monotonic()
                        stable_button = (
                            pending_button is not None and name == pending_button[0]
                            and max(abs(a-b) for a,b in zip(point, pending_button[1])) <= 3
                        )
                        if stable_button and now - last_button_time > 2:
                            position = (search["left"] + point[0], search["top"] + point[1])
                            if win.click(window, position):
                                LOG.info("已自動按下 %s。", name)
                                last_button_time = now
                                pending_button = None
                        else:
                            pending_button = button
                        time.sleep(0.12)
                        continue
                    pending_button = None
                location = find_board(screen)
                if location is None:
                    time.sleep(0.08)
                    continue
                x, y, width, height = location
                bounds = (search["left"] + x, search["top"] + y, width, height)
                LOG.info("已自動找到棋盤：左上角 (%d, %d)，大小 %d × %d。", *bounds)
            if region is None:
                left, top, width, height = bounds
                region = dict(left=left, top=top, width=width, height=height + round(height * 0.065))
                bar_start = height + round(height * 0.024)
                bar_end = height + round(height * 0.056)
            frame = capture.grab(region)
            board_frame = frame.crop(0, 0, width, height)
            now = time.monotonic()
            # Sample the left end of the bar, which stays filled until late in the round.
            if timer_active(frame.crop(width // 80, bar_start, width // 25 - width // 80, bar_end - bar_start)):
                if last_timer_seen is None:
                    LOG.info("已偵測到時間條。")
                    deadline = now + 63
                    idle_since = now
                last_timer_seen = now
            else:
                previous = None
                if last_timer_seen is not None and now - last_timer_seen > 1.4:
                    break
                time.sleep(0.03)
                continue
            if countdown_visible(board_frame):
                previous = None
                time.sleep(0.03)
                continue
            board = read_board(board_frame)
            flames = read_flames(board_frame, board)
            confirmed_flames = flames & previous_flames
            previous_flames = flames
            unknown = sum(row.count(None) for row in board)
            readable = unknown <= 2 and not runs(board)
            if readable and board == previous and board != last_move_board:
                move = best_move(board, confirmed_flames)
                if move is not None:
                    if dry_run:
                        print_board(board, confirmed_flames)
                        print(f"建議交換 {describe(move)}", flush=True)
                    else:
                        points = [
                            (left + int((c + 0.5) * width / SIZE), top + int((r + 0.5) * height / SIZE))
                            for r, c in move
                        ]
                        if not win.swap(window, *points):
                            previous = None
                            continue
                        moves += 1
                        LOG.debug("第 %d 次交換 %s；火焰格 %s；編號棋盤 %s", moves, describe(move), sorted(confirmed_flames), board)
                    last_move_board = board
                    idle_since = time.monotonic()
            previous = board if readable else None
            if not dry_run and now - idle_since > 6:
                if board == last_move_board:
                    raise ValueError("已送出交換，但六秒後棋盤仍相同。請確認遊戲仍在進行。")
                if unknown > 2:
                    raise ValueError(f"棋盤有 {unknown} 格無法辨識。請確認棋盤完整可見並重新框選。")
                raise ValueError("六秒內沒有找到可執行的走法。請確認框選範圍。")
            time.sleep(0.035)
    if last_timer_seen is None:
        raise ValueError("等待逾時，未找到完整棋盤。請確認遊戲已載入、畫面完整可見，或使用 --wait-key 手動開局。")
    LOG.info("本輪已停止。送出 %d 次交換，分數請查看遊戲結算畫面。", moves)
    return window


def configure_log():
    # 重新執行儲存格時關閉舊記錄，避免訊息重複或檔案被占用。
    for handler in LOG.handlers[:]:
        LOG.removeHandler(handler)
        handler.close()
    LOG.setLevel(logging.DEBUG)
    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    LOG.addHandler(console)
    try:
        log_file = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
    except OSError as error:
        LOG.warning("無法寫入診斷記錄：%s", error)
        return
    log_file.setFormatter(logging.Formatter("%(asctime)s %(message)s", datefmt="%H:%M:%S"))
    LOG.addHandler(log_file)
    LOG.info("診斷記錄存於 %s。", LOG_PATH)


def main(argv=None):
    parser = argparse.ArgumentParser(description="Bejeweled Blitz 棋盤辨識與自動交換")
    parser.add_argument("--dry-run", action="store_true", help="只列出辨識結果和建議走法")
    parser.add_argument("--manual", action="store_true", help="以 F2 手動框選棋盤")
    parser.add_argument("--wait-key", action="store_true", help="按 F2 待命並手動開局")
    parser.add_argument("--once", action="store_true", help="只玩一局後結束")
    parser.add_argument("--inspect", help="離線讀取一張截圖")
    parser.add_argument("--board", type=int, nargs=4, metavar=("X", "Y", "W", "H"))
    args = parser.parse_args(argv)
    if args.inspect:
        if not args.board:
            parser.error("--inspect 需要 --board X Y W H")
        image = read_image(args.inspect)
        x, y, width, height = args.board
        if min(x, y) < 0 or min(width, height) < 8 or (
            x + width > image.width or y + height > image.height
        ):
            parser.error("截圖或棋盤範圍無效")
        frame = image.crop(x, y, width, height)
        board = read_board(frame)
        flames = read_flames(frame, board)
        print_board(board, flames)
        if countdown_visible(frame):
            print("開局倒數遮住棋盤，略過走法。")
        elif any(None in row for row in board) or runs(board):
            print("棋盤尚未辨識完整或有寶石正在消除。")
        else:
            move = best_move(board, flames)
            print(f"建議交換 {describe(move)}" if move else "沒有找到合法交換。")
        return 0
    configure_log()
    try:
        window = play(args.dry_run, args.manual, args.wait_key)
        while not (args.once or args.dry_run or args.manual or args.wait_key):
            window = play(False, window=window)
    except KeyboardInterrupt:
        LOG.info("已停止。")
    except (OSError, ValueError) as error:
        LOG.error("停止原因：%s", error)
        return 1
    except Exception:
        LOG.exception("程式發生未預期的錯誤。")
        return 1
    return 0




## 5. 啟動

執行這一格後會控制滑鼠，自動遊玩直到按 F8 或中斷核心。

- 連續遊玩使用空清單 []
- 只玩一局改為 ["--once"]
- 手動開局改為 ["--wait-key"]，切到遊戲後按 F2，再用滑鼠開局


In [ ]:
執行參數 = []
結束代碼 = main(執行參數)
print(f"程式已結束，結束代碼為 {結束代碼}")
